# Prototype B: water stress scorecard
**Research question:** Do data centers stress water, and where is the build-out landing?

Water only for now. Each data center is placed in its river basin from its coordinates (`scripts/assign_basins.py`), so no county or zip step sits between the facility and the water data.

**Data:** WRI Aqueduct 4.0 baseline water stress (one 2023 snapshot); the project data center database; county population split across basins by overlap area (baseline for "where people live").
**What this can show:** whether data centers sit in more water-stressed basins than the country's population does (*siting*). **What it can't:** whether they changed water stress, or how much water they use (mostly unreported).
**Rating scale:** supported / not supported / mixed / can't tell. Correlation, not causation.

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name in ("data", "jppolizzi") else Path.cwd()
sys.path[:0] = [str(ROOT / ".pylibs"), str(ROOT / "scripts")]
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import proto_data as pdx
CLEAN = ROOT / "data" / "clean"
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportion_confint

ORDER = ["Low", "Low-medium", "Medium-high", "High", "Extremely high", "Arid, low use"]
HIGH = ["High", "Extremely high"]
d = pd.read_csv(CLEAN / "datacenters.csv").merge(pd.read_csv(CLEAN / "datacenters_basin.csv"), on="facility_id")
base = pd.read_csv(CLEAN / "basin_population.csv").set_index("stress_group")
d = d[d.coord_ok & d.status_group.isin(["operating", "pipeline"])]
no_basin = d.stress_group.isna() | d.stress_group.eq("No data")
print(f"{len(d)} operating or planned facilities with coordinates; {int(no_basin.sum())} have no basin data "
      f"(mostly metro Atlanta and parts of Virginia, where Aqueduct has gaps) and are left out")
d = d[~no_basin]
op = d[d.status_group == "operating"]; pl = d[d.status_group == "pipeline"]

## Where do facilities sit, compared with where people live?

In [ ]:
pct = lambda s: s.value_counts(normalize=True).reindex(ORDER).fillna(0) * 100
tab = pd.DataFrame({"operating facilities": pct(op.stress_group), "planned facilities": pct(pl.stress_group),
                    "US population": base["population_share"].reindex(ORDER) * 100, "US land area": base["land_share"].reindex(ORDER) * 100})
display(tab.round(1))

def high_share(df): return (df.stress_group.isin(HIGH)).sum(), len(df)
k, n = high_share(op); lo_, hi_ = proportion_confint(k, n, method="wilson")
pop_high = base.loc[HIGH, "population_share"].sum() * 100
print(f"Operating: {k / n * 100:.0f}% of facilities (95% CI {lo_ * 100:.0f}-{hi_ * 100:.0f}%, n={n}) are in High/Extremely high basins vs {pop_high:.0f}% of the US population")
k2, n2 = high_share(pl); lo2, hi2 = proportion_confint(k2, n2, method="wilson")
print(f"Planned:   {k2 / n2 * 100:.0f}% (95% CI {lo2 * 100:.0f}-{hi2 * 100:.0f}%, n={n2})")
mw = d.groupby(["stress_group", "status_group"])["mw"].sum().unstack().reindex(ORDER).fillna(0)
mw_pct = mw / mw.sum() * 100
display(mw.round(0))
print(f"Reported MW in High/Extremely high basins: operating {mw_pct.loc[HIGH, 'operating'].sum():.0f}%, planned {mw_pct.loc[HIGH, 'pipeline'].sum():.0f}%  (MW reported for {d.mw.notna().mean() * 100:.0f}% of facilities)")

## Scorecard figure (saved as an image)

In [ ]:
import textwrap
LAB = ["Low", "Low-\nmedium", "Medium-\nhigh", "High", "Extremely\nhigh", "Arid,\nlow use"]
fig = plt.figure(figsize=(13, 8.8))
gs = fig.add_gridspec(2, 2, height_ratios=[1.6, 0.8], hspace=0.55, wspace=0.25)
fig.suptitle("Do data centers stress water? Where is the build-out landing?  Prototype scorecard (water only)", fontsize=14, fontweight="bold", y=0.99)
x = np.arange(len(ORDER)); w = 0.27
ax = fig.add_subplot(gs[0, 0])
ax.bar(x - w, tab["US population"], w, color="#9ecae1", label="US population")
ax.bar(x, tab["operating facilities"], w, color="#1f5fa8", label=f"operating data centers (n={len(op)})")
ax.bar(x + w, tab["planned facilities"], w, color="#e8833a", label=f"planned data centers (n={len(pl)})")
ax.set_xticks(x); ax.set_xticklabels(LAB, fontsize=8.5); ax.set_ylabel("% of total"); ax.legend(fontsize=8, frameon=False)
ax.set_xlabel("baseline water stress of the river basin"); ax.set_title("1. Siting: data centers vs. where people live", fontsize=11)
ax = fig.add_subplot(gs[0, 1])
ax.bar(x - w / 2, mw_pct["operating"], w * 1.2, color="#1f5fa8", label=f"operating ({mw.operating.sum() / 1000:.0f} GW reported)")
ax.bar(x + w / 2, mw_pct["pipeline"], w * 1.2, color="#e8833a", label=f"planned ({mw.pipeline.sum() / 1000:.0f} GW reported)")
ax.set_xticks(x); ax.set_xticklabels(LAB, fontsize=8.5); ax.set_ylabel("% of reported MW"); ax.legend(fontsize=8, frameon=False)
ax.set_xlabel("baseline water stress of the river basin"); ax.set_title("2. Capacity: where the reported MW sits", fontsize=11)
ax = fig.add_subplot(gs[1, :]); ax.axis("off")
row = [["Water stress", f"facility in river basin\n(n={len(op)} operating,\n{len(pl)} planned)", f"{k / n * 100:.0f}% of operating data centers in\nHigh/Extremely high basins\n[{lo_ * 100:.0f}-{hi_ * 100:.0f}%] vs {pop_high:.0f}% of US population",
        "CAN'T TELL", "\n".join(textwrap.wrap("Snapshot shows siting, not change. Actual data center water use is mostly unreported; cooling type is known for few facilities. MW reported for under half of facilities.", 42))]]
t = ax.table(cellText=row, colLabels=["Concern", "Unit and n", "Estimate", "Rating", "Main caveat"], cellLoc="left", loc="upper center", colWidths=[0.1, 0.19, 0.27, 0.1, 0.34])
t.auto_set_font_size(False); t.set_fontsize(9.5); t.scale(1, 5.2)
for (i, j), cell in t.get_celld().items():
    if i == 0: cell.set_facecolor("#333"); cell.get_text().set_color("white"); cell.get_text().set_fontweight("bold")
    elif j == 3: cell.set_facecolor("#dddddd"); cell.get_text().set_fontweight("bold")
fig.text(0.5, 0.01, f"Correlation, not causation. Operating = status in database today (no build dates). {int(no_basin.sum())} facilities without basin data excluded. Source: WRI Aqueduct 4.0 (2023), project data center database.", ha="center", fontsize=8, style="italic")
(ROOT / "images").mkdir(exist_ok=True)
fig.savefig(ROOT / "images" / "prototype_b_scorecard.png", dpi=150, bbox_inches="tight", facecolor="white")
pd.DataFrame(row, columns=["concern", "unit and n", "estimate", "rating", "main caveat"]).to_csv(ROOT / "jppolizzi" / "prototype_b_scorecard.csv", index=False)
plt.show()

Next, one at a time: energy bills, then grid strain. Open decisions: whether to find build dates so growth can be measured; filling the Aqueduct gaps around Atlanta.